In [1]:
########################################################################################
# This creates the base "Notam Map.html" or user defined name for one-off runs
# Notam Map.html is the basemap, and will pull NOTAM data for /data/Notams/*.json files
#
########################################################################################

### Globl Imports
import html
import json
import math
import re
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import FeatureGroupSubGroup, GroupedLayerControl, MarkerCluster

In [2]:
# ==============================
# GLOBAL CONFIG collections
# ==============================
DEFAULT_ZOOM = 10
NM_TO_M = 1852
NOTAM_RADIUS_KM = 10
NOTAM_RADIUS_M = NOTAM_RADIUS_KM * 1000
COORD_DECIMALS = 6

In [3]:
# Point this to your local GitHub repo clone root
REPO_ROOT = Path(r"C:\Users\Ron Treleaven\Drone_SOP")
NOTAMS_FILTERED_PATH = REPO_ROOT / "data" / "notams" / "All_CA.json"
NOTAMS_RAW_PATH = REPO_ROOT / "data" / "notams" / "All_CA_raw.json"
META_PATH = REPO_ROOT / "data" / "notams" / "All_CA.meta.json"

In [4]:
print(REPO_ROOT)

C:\Users\Ron Treleaven\Drone_SOP


In [5]:
ICON_STYLE = {
    "heliport": {"icon": "helicopter", "color": "green"},
    "seaplane_base": {"icon": "ship", "color": "cadetblue"},
    "small_airport": {"icon": "circle", "color": "gray"},
    "medium_airport": {"icon": "plane", "color": "blue"},
    "large_airport": {"icon": "plane", "color": "darkblue"},
    "default": {"icon": "map-marker", "color": "lightgray"},
}

In [6]:
## to define a radius of NOTAMS by pilot lat/lon

def haversine_km(lat1, lon1, lat2, lon2):
    r_km = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)

    a = (
        math.sin(d_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    )
    return 2 * r_km * math.atan2(math.sqrt(a), math.sqrt(1 - a))


In [7]:
def format_notam_raw_for_popup(raw_text):
    compact = re.sub(r"\s+", " ", (raw_text or "")).strip()
    if not compact:
        return ""

    section_split = r"(?=\b(?:Q|A|B|C|D|E|F|G|X)\))"
    parts = [part.strip() for part in re.split(section_split, compact) if part.strip()]
    return "<br>".join(html.escape(part) for part in parts)


Get location code TODO

Unreliable GEOLOCATOR from code...  The HTML page does a better job, but this gives an auto 
replace GEOLOCATOR method, to prompt for PC or GPS values..



In [8]:
def get_pilot_location():
    print("Pilot coordinate format: Decimal Degrees (DD)")
    print("Example latitude: 43.653200")
    print("Example longitude: -79.383200")

    fetch_location = input(
        "Would you like to fetch your current location? (y/n): "
    ).strip().lower()

    if fetch_location == "y":
        try:
            import geocoder

            print("Fetching your current location...")
            g = geocoder.ip("me")

            if g.ok and g.latlng and len(g.latlng) == 2:
                pilot_lat, pilot_lon = g.latlng
                print(f"Detected: {pilot_lat}, {pilot_lon}")

                confirm = input("Use this location? (y/n): ").strip().lower()
                if confirm == "y":
                    return float(pilot_lat), float(pilot_lon)

            print("Falling back to manual entry.")
        except ImportError:
            print("geocoder package not installed.")
            print("Falling back to manual entry.")
        except Exception as exc:
            print(f"Unable to auto-detect location: {exc}")
            print("Falling back to manual entry.")

    while True:
        try:
            lat = float(
                input("Pilot latitude DD (e.g., 43.653200, range -90 to 90): ").strip()
            )
            lon = float(
                input("Pilot longitude DD (e.g., -79.383200, range -180 to 180): ").strip()
            )
            if not (-90 <= lat <= 90):
                print("Latitude out of range. Use -90 to 90.")
                continue
            if not (-180 <= lon <= 180):
                print("Longitude out of range. Use -180 to 180.")
                continue
            return lat, lon
        except ValueError:
            print("Numeric DD lat/lon required (example: 43.653200 and -79.383200).")


In [9]:
get_pilot_location()

Pilot coordinate format: Decimal Degrees (DD)
Example latitude: 43.653200
Example longitude: -79.383200


Would you like to fetch your current location? (y/n):  n
Pilot latitude DD (e.g., 43.653200, range -90 to 90):  44
Pilot longitude DD (e.g., -79.383200, range -180 to 180):  -79


(44.0, -79.0)

In [10]:
def load_notams_file(path, label):
    if not path.exists():
        raise SystemExit(f"NOTAM file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise SystemExit(f"{label} is not a JSON array.")

    return data

In [11]:
def load_notams_from_repo():
    filtered_data = load_notams_file(NOTAMS_FILTERED_PATH, "All_CA.json")
    raw_data = load_notams_file(NOTAMS_RAW_PATH, "All_CA_raw.json")

    generated_at = None
    if META_PATH.exists():
        try:
            with META_PATH.open("r", encoding="utf-8") as f:
                meta = json.load(f)
            generated_at = meta.get("generatedAt")
        except Exception:
            generated_at = None

    return filtered_data, raw_data, generated_at



In [12]:
def prepare_obstacle_points(notams_data):
    points = []
    for notam in notams_data:
        coords = notam.get("coordinates_dd") or []
        raw = (notam.get("raw") or "").strip()
        if not coords or not raw:
            continue

        start_val = str(notam.get("startValidity") or "N/A")
        end_val = str(notam.get("endValidity") or "N/A")
        safe_raw = format_notam_raw_for_popup(raw)

        for coord in coords:
            try:
                lat, lon = map(float, coord.split(","))
            except Exception:
                continue
            points.append(
                {
                    "lat": lat,
                    "lon": lon,
                    "start": start_val,
                    "end": end_val,
                    "rawHtml": safe_raw,
                }
            )

    return points



In [13]:
def build_map(pilot_lat, pilot_lon, notams_filtered, notams_raw, generated_at=None):
    # 1) Airports
    df = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv")
    df = df[df["iso_country"] == "CA"].copy()

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude_deg, df.latitude_deg),
        crs="EPSG:4326",
    )

In [15]:
    # 2) Base map
    m = folium.Map(
        location=[pilot_lat, pilot_lon],
        zoom_start=DEFAULT_ZOOM,
        tiles="CartoDB positron",
        zoom_control=False,
    )



NameError: name 'pilot_lat' is not defined